In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 10.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchmetrics]


In [2]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, Subset
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models import resnet50, ResNet50_Weights
import time
import torchmetrics
from torchmetrics.classification import Accuracy, F1Score
from tqdm import tqdm


In [3]:
class MNISTDetectionDataset(Dataset):
    def __init__(self, root='./data', train=True):
        self.mnist = torchvision.datasets.MNIST(
            root=root,
            train=train,
            download=True
        )

        # Transformation : redimensionnement (ex: 128x128) et conversion en 3 canaux RGB
        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(), # Renvoie un tenseur de forme [3, 128, 128]
        ])

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        img, label = self.mnist[idx]
        img_tensor = self.transform(img) # Forme [3, 128, 128]

        _, h, w = img_tensor.shape
        boxes = torch.tensor([[0, 0, w, h]], dtype=torch.float32)

        # Étant donné que Faster R-CNN utilise la classe 0 pour le fond,
        # on décale les classes de MNIST (0-9 -> 1-10)
        target_label = torch.tensor([label + 1], dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": target_label
        }

        return img_tensor, target

# Collate function indispensable pour regrouper les listes de dictionnaires en batchs
def collate_fn(batch):
    return tuple(zip(*batch))

# Instanciation propre des datasets
train_dataset = MNISTDetectionDataset(train=True)
test_dataset = MNISTDetectionDataset(train=False)

# Par exemple, ne garder que les 3000 premiers exemples d'entraînement et 500 de test
train_subset_indices = range(3000)
test_subset_indices = range(500)

train_dataset_small = Subset(train_dataset, train_subset_indices)
test_dataset_small = Subset(test_dataset, test_subset_indices)

train_loader = torch.utils.data.DataLoader(
    train_dataset_small, batch_size=8, shuffle=True, collate_fn=collate_fn
)
test_loader = torch.utils.data.DataLoader(
    test_dataset_small, batch_size=8, shuffle=False, collate_fn=collate_fn
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 496kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.61MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.0MB/s]


In [4]:
def get_faster_rcnn_model(num_classes=11): # 10 chiffres + 1 classe background (fond)
    # Chargement d'un backbone ResNet-50 pré-entraîné
    backbone = torchvision.models.resnet50(weights=ResNet50_Weights.DEFAULT)
    # On retire les couches de classification finales pour garder uniquement l'extracteur de features
    modules = list(backbone.children())[:-2]
    backbone = torch.nn.Sequential(*modules)
    backbone.out_channels = 2048 # Nombre de canaux de sortie pour ResNet-50

    # Configuration des ancres pour le RPN
    anchor_generator = AnchorGenerator(
        sizes=((32, 64, 128, 256, 512),),
        aspect_ratios=((0.5, 1.0, 2.0),)
    )

    # ROI Pooler pour redimensionner les régions d'intérêt
    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=['0'],
        output_size=7,
        sampling_ratio=2
    )

    # Assemblage du modèle Faster R-CNN complet
    model = FasterRCNN(
        backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler
    )

    return model

In [5]:
def train_and_evaluate_faster_rcnn(model, train_loader, test_loader, optimizer, num_epochs=5, device='cuda'):
    model.to(device)

    # 1. ENTRAÎNEMENT
    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

        for images, targets in tqdm(train_loader):
            # images est un tuple de tenseurs [3, 128, 128], on les envoie sur le device sous forme de liste
            images = [img.to(device) for img in images]

            # Formatage des cibles
            formatted_targets = []
            for t in targets:
                target_dict = {
                    "boxes": t["boxes"].to(device),
                    "labels": t["labels"].to(device)
                }
                formatted_targets.append(target_dict)

            optimizer.zero_grad()
            loss_dict = model(images, formatted_targets)
            losses = sum(loss for loss in loss_dict.values())

            losses.backward()
            optimizer.step()

            epoch_loss += losses.item()

        print(f"Train Loss (Faster R-CNN): {epoch_loss / len(train_loader):.4f}")

    training_time = time.time() - start_time
    print(f"Temps d'entraînement total : {training_time:.2f} secondes")

    # 2. ÉVALUATION (Calcul Accuracy, F1-Score, Loss de test)
    model.eval()
    test_acc_metric = Accuracy(task="multiclass", num_classes=11).to(device)
    test_f1_metric = F1Score(task="multiclass", num_classes=11, average='macro').to(device)

    test_acc_metric.reset()
    test_f1_metric.reset()

    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc="Évaluation"):
            images = [img.repeat(3, 1, 1).to(device) for img in images]

            # Prédictions du modèle
            predictions = model(images)

            # Extraction de la classe prédite avec le score le plus élevé pour chaque image
            preds_list = []
            targets_list = []

            for i, pred in enumerate(predictions):
                if len(pred['labels']) > 0:
                    best_pred_label = pred['labels'][0] # Label avec la plus haute confiance
                else:
                    best_pred_label = torch.tensor(0, device=device) # Background par défaut si aucune boîte détectée
                preds_list.append(best_pred_label)

                # Récupération sécurisée du label cible depuis le dictionnaire t
                true_label = targets[i]['labels']
                if isinstance(true_label, torch.Tensor):
                    true_label = true_label.item()

                # Cible réelle
                targets_list.append(torch.tensor(true_label, device=device))

            if preds_list and targets_list:
                preds_tensor = torch.stack(preds_list)
                targets_tensor = torch.stack(targets_list)
                test_acc_metric.update(preds_tensor, targets_tensor)
                test_f1_metric.update(preds_tensor, targets_tensor)

    final_test_acc = test_acc_metric.compute()
    final_test_f1 = test_f1_metric.compute()

    print("\n--- Résultats d'Évaluation (Faster R-CNN) ---")
    print(f"Accuracy (Test) : {final_test_acc.item():.4f}")
    print(f"F1-Score (Test) : {final_test_f1.item():.4f}")
    print(f"Temps total     : {training_time:.2f} secondes")

    return {
        'model': 'Faster R-CNN',
        'accuracy': final_test_acc.item(),
        'f1_score': final_test_f1.item(),
        'training_time': training_time
    }

In [6]:
# 1. Détection du périphérique (GPU/CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilisation du périphérique : {device}")

# 2. Instanciation du modèle Faster R-CNN (10 chiffres + 1 classe fond)
model = get_faster_rcnn_model(num_classes=11)

# 3. Configuration de l'optimiseur
# (Pour Faster R-CNN, SGD avec momentum et weight decay est le choix standard recommandé)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# 4. Lancement effectif de l'entraînement et de l'évaluation
# (On met 2 ou 3 époques pour commencer, car Faster R-CNN est lourd à entraîner)
history_faster_rcnn = train_and_evaluate_faster_rcnn(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    num_epochs=3,
    device=device
)

Utilisation du périphérique : cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 114MB/s]



Epoch [1/3]


100%|██████████| 375/375 [15:09<00:00,  2.43s/it]


Train Loss (Faster R-CNN): 0.3260

Epoch [2/3]


100%|██████████| 375/375 [15:16<00:00,  2.44s/it]


Train Loss (Faster R-CNN): 0.0999

Epoch [3/3]


100%|██████████| 375/375 [15:17<00:00,  2.45s/it]


Train Loss (Faster R-CNN): 0.0668
Temps d'entraînement total : 2744.39 secondes


Évaluation:   0%|          | 0/63 [00:00<?, ?it/s]


RuntimeError: The size of tensor a (9) must match the size of tensor b (3) at non-singleton dimension 0